In [5]:
import ev_sim

experiment = "ann-arbor-fleet-sizing-2seaters"
ev_sim.generate_runs_from_yaml(f"experiments/{experiment}.yaml")

[{'run_name': 'run_0000',
  'config_path': PosixPath('runs/ann-arbor-fleet-sizing-2seaters/run_0000/config.yaml'),
  'config': {'sim': {'region': 'ann-arbor',
    'length_s': 604800,
    'h3_resolution': 7,
    'requests_file': 'data/requests/ann-arbor.parquet',
    'riders_file': 'data/riders/ann-arbor-3000.parquet',
    'output': 'runs/ann-arbor-fleet-sizing-2seaters/run_0000/raw',
    'cos_ref_lat': 0.739953730674639},
   'policy': {'matching': 'bounded-h3',
    'max_response_time_s': 900,
    'charging': 'hubs',
    'n_hubs': 2,
    'charging_hubs': 'data/charging-hubs/ann-arbor-2.csv'},
   'fleet': {'frac_2_seater': 1.0, 'frac_4_seater': 0.0, 'fleet_size': 3000},
   'name': 'ann-arbor-fleet-sizing-2seaters'},
  'run_dir': PosixPath('runs/ann-arbor-fleet-sizing-2seaters/run_0000')},
 {'run_name': 'run_0001',
  'config_path': PosixPath('runs/ann-arbor-fleet-sizing-2seaters/run_0001/config.yaml'),
  'config': {'sim': {'region': 'ann-arbor',
    'length_s': 604800,
    'h3_resolution'

In [6]:
from pathlib import Path
import yaml
import pandas as pd
import ev_sim

experiment = "ann-arbor-fleet-sizing-2seaters"
manifest = Path("runs") / experiment / "manifest.txt"

# Use only runs listed in the current manifest.
run_dirs = [
    Path(line.strip())
    for line in manifest.read_text().splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]

print(f"Current manifest contains {len(run_dirs)} runs.")

if len(run_dirs) != 15:
    raise ValueError(
        "Your posted YAML defines 15 runs. "
        "Check that this manifest matches that YAML."
    )

# Check all current outputs before calculating statistics.
missing = [
    str(folder / "raw" / "requests.parquet")
    for folder in run_dirs
    if not (folder / "raw" / "requests.parquet").is_file()
]

if missing:
    raise FileNotFoundError(
        "These current runs have missing outputs:\n" + "\n".join(missing)
    )

rows = []

for folder in run_dirs:
    config = yaml.safe_load((folder / "config.yaml").read_text())

    inputs = pd.read_parquet(
        config["sim"]["requests_file"],
        columns=["created_at", "pax"],
    )
    output = pd.read_parquet(folder / "raw" / "requests.parquet")

    requests = inputs.merge(
        output,
        left_index=True,
        right_index=True,
        validate="1:1",
    )

    if len(requests) != len(output):
        raise ValueError(f"Input/output request indices do not match: {folder}")

    stats = ev_sim.compute_request_statistics(requests)
    rows.append({
        "fleet.fleet_size": config["fleet"]["fleet_size"],
        **stats,
    })

df = pd.DataFrame(rows).sort_values("fleet.fleet_size")

columns = [
    "fleet.fleet_size",
    "total",
    "completed",
    "dropped",
    "service_level",
    "mean_response_time",
    "q1_response_time",
    "median_response_time",
    "mean_response_time",
    "q3_response_time",
    "pct90_response_time",
    "pct95_response_time",
]

print(df[columns].T.to_string())

Current manifest contains 15 runs.
                                0             1             2             3             4             5             6             7             8             9             10            11            12            13            14
fleet.fleet_size      3.000000e+03  3.500000e+03  4.000000e+03  4.500000e+03  5.000000e+03  5.500000e+03  6.000000e+03  6.500000e+03  7.000000e+03  7.500000e+03  8.000000e+03  8.500000e+03  9.000000e+03  9.500000e+03  1.000000e+04
total                 2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06  2.441624e+06
completed             1.704723e+06  1.884194e+06  2.022613e+06  2.195410e+06  2.252682e+06  2.258850e+06  2.258857e+06  2.258857e+06  2.258857e+06  2.258857e+06  2.258857e+06  2.258857e+06  2.258857e+06  2.258857e+06  2.258857e+06
dropped               7.369010e+05  5.574